# 04 · FinSight demo app

Upload a digital financial PDF (up to 3 pages) and ask a question. The app shows the answer, the formula Python executed, the verifier checks, and the page with the evidence rows highlighted.

Test PDFs: any file from `fp/dataset/tatdqa_docs_test/test/*.pdf` on your laptop.

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
from finsight.infer import load_model, predict
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def latest_adapter(run):
    """The finished adapter, or the newest checkpoint if training stopped midway."""
    import glob
    if os.path.exists(f"{run}/final"):
        return f"{run}/final"
    ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    assert ckpts, f"No adapter in {run} - run notebook 02 (training) first"
    print("NOTE: training has not finished; using", ckpts[-1])
    return ckpts[-1]
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
SFT_RUN = f"{OUT}/sft_Qwen2.5-3B-Instruct"
ADAPTER = latest_adapter(SFT_RUN)
model, tok = load_model(MODEL_ID, adapter=ADAPTER)
# To demo the expert-iteration model instead:
# model, tok = load_model(MODEL_ID, adapter=latest_adapter(f"{OUT}/ei1_Qwen2.5-3B-Instruct"))

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
import gradio as gr
from finsight.demo import doc_from_pdf, answer, highlight_pages, explain_html

def run(pdf, question):
    if pdf is None or not question.strip():
        return "<p>Upload a PDF and type a question.</p>", []
    path = pdf if isinstance(pdf, str) else pdf.name
    doc, scales = doc_from_pdf(path)
    res = answer(model, tok, doc, question)
    return explain_html(res, doc), highlight_pages(path, doc, scales, res["parsed"]["evidence"])

app = gr.Interface(
    fn=run,
    inputs=[gr.File(label="Financial PDF (digital, up to 3 pages)", file_types=[".pdf"]),
            gr.Textbox(label="Question", placeholder="What was the percentage change in revenue from 2018 to 2019?")],
    outputs=[gr.HTML(label="Answer"), gr.Gallery(label="Evidence on the page", columns=1, height=800)],
    title="FinSight: grounded question answering on financial documents",
    description="The model cites evidence rows and writes a formula; Python computes the answer.")
app.launch(share=True)   # opens inside the notebook; the public link works for about 72 h